# Gauge Reader — train the models on Kaggle (2 × T4)

Trains the three models (docs/PLAN.md §4) using **both GPUs at once**, then evaluates the full system on
synth-test (E2). This notebook only *calls the repo's scripts* (CLAUDE.md: no notebook-only logic).

| GPU | Jobs, in order |
|---|---|
| 0 | Model A — detector + segmenter (`models.train_seg`) |
| 1 | Model B — keypoints (`models.train_pose`), then Model C — regression for S3 (`models.train_regress`) |

Two independent jobs (one per GPU) rather than one DDP job: the models are small and Kaggle has 4 CPU cores,
so splitting one model over two GPUs mostly adds overhead.

**Expected wall time** (estimates, not yet measured; data loading on 4 CPUs may stretch them):

| Mode | Settings | Training | Whole notebook |
|---|---|---|---|
| `FAST = True` (default) | seg 20 epochs @ 512 px · pose 40 epochs · regression 10 epochs | ~1–1.3 h | **~1.5 h** |
| `FAST = False` | spec defaults: seg 60 @ 640 · pose 100 · regression 30 | ~3–4.5 h | **~3.5–5 h** |

Both fit in one Kaggle session (12 h limit).

### Setup (once)

1. **Datasets → New Dataset**: upload `cv-dial-src.zip` and `cv-dial-data.zip` (from the repo's `data/` folder).
   Kaggle may unpack the zips; the notebook works either way.
2. In this notebook: **Add Input** → your dataset.
3. **Settings → Accelerator: GPU T4 x2**, and **Internet: On** (needed for `pip` and the pretrained weights;
   requires a phone-verified Kaggle account).
4. Run interactively, or **Save Version → Save & Run All** to run in the background with the browser closed.
   Weights, metrics and plots appear in the notebook's **Output** tab (`results/`, plus `cv-dial-results.zip`).

## 1. Settings

In [ ]:
FAST       = True      # True: short schedule (see table above) | False: spec defaults
RUN_SMOKE  = True      # ~1-2 min GPU smoke test before the long jobs
WORKERS    = 2         # data-loader workers per job (Kaggle has 4 CPUs, two jobs run at once)
POSE_CACHE = "ram"     # cache pose crops in RAM (~10 GB) to relieve the CPUs; "" to disable
COPY_THREADS = 64      # parallel file copies from /kaggle/input (slow per file) to the local disk
INPUT      = "/kaggle/input"
REPO       = "/tmp/cv-dial"                    # working copy (not saved as notebook output)
OUT        = "/kaggle/working/results"         # saved as notebook output

## 2. GPUs

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch, sys
N_GPU = torch.cuda.device_count()
print("python", sys.version.split()[0], "| torch", torch.__version__, "| GPUs:", N_GPU)
assert N_GPU >= 1, "No GPU: Settings -> Accelerator -> GPU T4 x2"
if N_GPU == 1:
    print("Only one GPU: the jobs will run one after another on GPU 0.")

## 3. Code and dependencies

In [ ]:
import os, shutil, subprocess

def find(root, test, max_depth=4):
    """First path under root (breadth-first, shallow) for which test(path) is true."""
    level = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in level:
            try:
                entries = sorted(os.scandir(d), key=lambda e: e.name)
            except (NotADirectoryError, PermissionError, FileNotFoundError):
                continue
            for e in entries:
                if test(e.path):
                    return e.path
                if e.is_dir(follow_symlinks=False) and e.name not in ("images", "labels", "records"):
                    nxt.append(e.path)
        level = nxt
    return None

if os.path.exists(REPO):
    shutil.rmtree(REPO)
src_dir = find(INPUT, lambda p: os.path.isfile(os.path.join(p, "gen", "generate.py")))
src_zip = find(INPUT, lambda p: os.path.basename(p) == "cv-dial-src.zip")
if src_dir:
    shutil.copytree(src_dir, REPO)
elif src_zip:
    subprocess.run(["unzip", "-q", src_zip, "-d", REPO], check=True)
else:
    raise FileNotFoundError("cv-dial code not found under /kaggle/input: add the dataset with cv-dial-src.zip")
print("code from", src_dir or src_zip)
%cd {REPO}
# Kaggle ships torch, torchvision, OpenCV, NumPy, matplotlib, Pillow, PyYAML; only Ultralytics is added.
!pip install -q "ultralytics==8.4.178"
!python -c "import ultralytics, cv2; print('ultralytics', ultralytics.__version__, '| opencv', cv2.__version__)"

## 4. Data and results

`/kaggle/input` is slow per file (~0.2 MB/s, see Ultralytics' "Slow image access" warning): training straight
from it spends over an hour just scanning 92k small files. So the data is first copied to the local disk with
many parallel copies (a few minutes; progress is printed), or unpacked there if Kaggle kept the zip as a zip.
Local data is also writable, so Ultralytics can save its label cache. `results/` is linked to
`/kaggle/working/results`, so it is saved with the notebook.

In [ ]:
import os, subprocess, time
t0 = time.time()
has_data = lambda p: os.path.isdir(os.path.join(p, "synth_seg")) and os.path.isdir(os.path.join(p, "synth_crops"))
LOCAL = "/tmp/cv-dial-data"

def parallel_copy(src, dst, threads):
    """Copy a directory tree with many concurrent file copies (hides /kaggle/input's per-file latency)."""
    from concurrent.futures import ThreadPoolExecutor
    files = []
    for root, _, names in os.walk(src, followlinks=True):
        rel = os.path.relpath(root, src)
        os.makedirs(os.path.join(dst, rel), exist_ok=True)
        files += [os.path.join(rel, n) for n in names]
    t, n = time.time(), len(files)
    copy = lambda rel: shutil.copyfile(os.path.join(src, rel), os.path.join(dst, rel))
    with ThreadPoolExecutor(threads) as ex:
        for i, _ in enumerate(ex.map(copy, files), 1):
            if i % 5000 == 0 or i == n:
                rate = i / (time.time() - t)
                print(f"  {i:6d}/{n} files  {rate:6.0f} files/s  ~{(n - i) / rate / 60:4.1f} min left", flush=True)

data_root = find(LOCAL, has_data) if os.path.isdir(LOCAL) else None     # already copied in this session
if data_root is None:
    src_root = find(INPUT, has_data)
    if src_root is not None:
        print("copying", src_root, "->", LOCAL)
        for name in ("synth_seg", "synth_crops", "synth"):
            parallel_copy(os.path.join(src_root, name), os.path.join(LOCAL, "data", name), COPY_THREADS)
    else:
        data_zip = find(INPUT, lambda p: os.path.basename(p) == "cv-dial-data.zip")
        if data_zip is None:
            raise FileNotFoundError("data not found under /kaggle/input: add the dataset with cv-dial-data.zip")
        print("unzipping", data_zip, "->", LOCAL)
        subprocess.run(["unzip", "-q", "-o", data_zip, "-d", LOCAL], check=True)
    data_root = find(LOCAL, has_data)
print("data from", data_root)
os.makedirs("data", exist_ok=True)
for name in ("synth_seg", "synth_crops", "synth"):
    link = os.path.join("data", name)
    if os.path.lexists(link):
        os.remove(link) if os.path.islink(link) else shutil.rmtree(link)
    os.symlink(os.path.join(data_root, name), link)
os.makedirs(OUT, exist_ok=True)
if os.path.lexists("results"):
    os.remove("results") if os.path.islink("results") else shutil.rmtree("results")
os.symlink(OUT, "results")
for d in ["data/synth_seg/images/train", "data/synth_seg/images/val", "data/synth_crops/images/train",
          "data/synth_crops/images/val", "data/synth/test/records"]:
    print(f"{d:35s} {len(os.listdir(d)) if os.path.isdir(d) else 'MISSING'}")
print(f"ready in {time.time() - t0:.0f} s")

## 5. Smoke test (optional, ~1–2 min)
Catches environment problems before the long jobs. Its numbers mean nothing.

In [ ]:
if RUN_SMOKE:
    !CUDA_VISIBLE_DEVICES=0 python -m models.train_pose --config configs/train/pose.yaml --smoke --set train.device=0 2>&1 | tail -3

## 6. Train on both GPUs

Starts the jobs in the background (one per GPU) and returns immediately; step 7 waits for them.
Each run writes `results/<name>_<timestamp>/` (resolved `config.yaml`, `run.json`, weights); console output goes
to `results/logs/gpu<N>.log`. `fliplr: 0.0` is enforced for pose and regression (CLAUDE.md invariant 2).

In [ ]:
import subprocess, os, shlex
fast = {"seg": ["train.epochs=20", "train.imgsz=512"],
        "pose": ["train.epochs=40", "train.patience=15"],
        "regress": ["train.epochs=10"]} if FAST else {"seg": [], "pose": [], "regress": []}
extra = {"seg": [f"train.workers={WORKERS}"],
         "pose": [f"train.workers={WORKERS}"] + ([f"train.cache={POSE_CACHE}"] if POSE_CACHE else []),
         "regress": [f"train.workers={WORKERS}"]}

def train_cmd(name):
    sets = " ".join(shlex.quote(s) for s in fast[name] + extra[name])
    return f"python -m models.train_{name} --config configs/train/{name}.yaml --set {sets}"

queues = {"0": ["seg"], "1": ["pose", "regress"]} if N_GPU >= 2 else {"0": ["seg", "pose", "regress"]}
os.makedirs("results/logs", exist_ok=True)
procs = {}
for gpu, names in queues.items():
    script = " && ".join(train_cmd(n) for n in names)
    log = open(f"results/logs/gpu{gpu}.log", "w")
    procs[gpu] = subprocess.Popen(["bash", "-c", script], stdout=log, stderr=subprocess.STDOUT,
                                  env={**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "PYTHONUNBUFFERED": "1"})
    print(f"GPU {gpu}: {' -> '.join(names)}  (pid {procs[gpu].pid}, log results/logs/gpu{gpu}.log)")

## 7. Wait for both GPUs

Every 2 minutes, prints each GPU's latest log line (long paths shortened), or "(no new output)" if the job
printed nothing since the last check. Safe to interrupt and re-run: it only *watches*, the jobs keep running.

In [ ]:
import time, re
start = time.time()

def last_line(path):
    try:
        text = open(path, errors="replace").read()[-4000:]
    except FileNotFoundError:
        return ""
    parts = [p.strip() for p in re.split(r"[\r\n]", text) if p.strip()]
    if not parts:
        return ""
    line = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", parts[-1])
    line = re.sub(r"(/[^\s/]+)+/", ".../", line)          # shorten paths so the progress numbers stay visible
    line = re.sub(r"[━─]{3,}", "", line)                  # drop progress-bar glyphs
    return line[:160]

previous = {}
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    print(f"--- {(time.time() - start) / 60:5.1f} min")
    for gpu, p in procs.items():
        state = "running" if p.poll() is None else f"done (exit {p.returncode})"
        line = last_line(f"results/logs/gpu{gpu}.log")
        shown = line if line != previous.get(gpu) else "(no new output)"
        previous[gpu] = line
        print(f"GPU {gpu} [{state}] {shown}")

failed = {g: p.returncode for g, p in procs.items() if p.returncode != 0}
for gpu in failed:
    print(f"\n===== GPU {gpu} failed; last lines of results/logs/gpu{gpu}.log =====")
    !tail -n 30 results/logs/gpu{gpu}.log
assert not failed, f"training failed: {failed}"
print(f"all training finished in {(time.time() - start) / 60:.1f} min")

## 8. Latest trained weights

In [ ]:
import glob, json, os
def latest(name):
    """Newest finished, non-smoke run of `name` that has weights."""
    runs = []
    for p in glob.glob(f"results/{name}_*/run.json"):
        info = json.load(open(p))
        if "_smoke_" not in p and info.get("status") == "done" and os.path.exists(info.get("weights", "")):
            runs.append((os.path.getmtime(p), info["weights"]))
    return max(runs)[1] if runs else None

W = {k: latest(k) for k in ("seg", "pose", "regress")}
for k, v in W.items():
    print(f"{k:8s} {v or 'not trained yet'}")

## 9. E2 — full system on synth-test

Every reported number comes from `eval.evaluate` (CLAUDE.md invariant 7). Runs the pipeline on GPU 0 via
`configs/eval/pipeline_gpu.yaml`.

In [ ]:
import time
assert all(W.values()), f"train all three models first: {W}"
RUN = f"results/e2_{time.strftime('%Y%m%d-%H%M%S')}"
SEG_W, POSE_W, REG_W = W["seg"], W["pose"], W["regress"]
!python -m eval.evaluate --run {RUN} --split synth-test --seg-weights {SEG_W} --pose-weights {POSE_W} --regress-weights {REG_W} --set pipeline=configs/eval/pipeline_gpu.yaml
!python -m eval.plots run --run {RUN}

In [ ]:
from IPython.display import Image, display
display(Image(f"{RUN}/plots/mae_vs_obliquity.png"))
display(Image(f"{RUN}/plots/mae_bins.png"))

## 10. Package the results

In [ ]:
# One download with every finished run (weights, configs, metrics, plots); smoke runs are left out.
!cd /kaggle/working && zip -q -r cv-dial-results.zip results -x "results/*_smoke_*" && ls -lh cv-dial-results.zip

## 11. Use the weights locally

Download `cv-dial-results.zip` from the **Output** tab, unzip it into the repo root (it contains `results/`), then:

```bash
uv run python -m pipeline.demo_app \
    --seg-weights results/<seg_run>/ultralytics/weights/best.pt \
    --pose-weights results/<pose_run>/ultralytics/weights/best.pt \
    --regress-weights results/<regress_run>/weights/best.pt
```

The exact paths are printed in step 8. E3 (real photos) runs the same way with `--split real` once
`data/real/` exists; it needs no GPU.